# Funciones de activación

Cada neurona calcula un puntaje $z = w \cdot x + b$ y después le aplica una función $g$: la **función de activación**. Esa $g$ es la que decide qué forma tiene la salida de la neurona y, sobre todo, la que impide que la red entera sea una sola recta.

Hasta ahora se usó `sigmoid` en todas las neuronas. Este lab explica por qué eso no siempre es correcto, presenta la opción lineal y ReLU, y da la regla práctica para elegir en la capa de salida y en las capas ocultas.

### Por qué `sigmoid` en todas no funciona

Dos problemas concretos:

1. **Acota la salida a $(0, 1)$.** Sirve para probabilidades, pero no para predecir un precio de 250 mil dólares ni un cambio de precio negativo. Si la última neurona es sigmoide, la red literalmente no puede expresar esos valores.
2. **Se satura.** Para $z$ muy grande o muy chico, la curva se aplana y su derivada tiende a cero. En el gradiente descendiente eso significa pasos cada vez más chicos: el aprendizaje se estanca (el llamado *vanishing gradient*).

Por eso `sigmoid` es una buena elección para la salida de una clasificación binaria, pero no un valor por defecto para cualquier neurona.

### La activación lineal: no usar activación

La opción lineal es simplemente $g(z) = z$: la neurona devuelve el puntaje sin transformarlo. En Keras se pide con `activation="linear"` (o `activation=None`).

No es inútil: es exactamente lo que se necesita cuando la salida debe poder tomar cualquier valor real, como en una regresión de precios. Lo que no puede hacerse es usar solo lineales en toda la red (eso se ve al final).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Estilo para las gráficas
plt.style.use('ggplot')

def sigmoid(z):
    """Calcula la función sigmoide g(z) = 1 / (1 + e^(-z))."""
    return 1 / (1 + np.exp(-z))

def relu(z):
    """Calcula ReLU: el máximo entre 0 y z."""
    return np.maximum(0, z)

# Las tres funciones sobre el mismo rango de puntajes
z = np.linspace(-6, 6, 400)
plt.plot(z, sigmoid(z), c='purple', label='sigmoid: acota a (0, 1)')
plt.plot(z, z, c='b', label='lineal: g(z) = z')
plt.plot(z, relu(z), c='g', label='ReLU: max(0, z)')
plt.axhline(0, c='k', linestyle=':', linewidth=1)
plt.axvline(0, c='k', linestyle=':', linewidth=1)
plt.title("Tres activaciones, tres formas de salida")
plt.xlabel("Puntaje z")
plt.ylabel("Activación g(z)")
plt.legend()
plt.show()

# Los mismos tres puntajes en cada función
for z_ej in [-2.0, 0.0, 2.0]:
    print(f"z = {z_ej:+.1f}: sigmoid = {sigmoid(z_ej):.3f} | lineal = {z_ej:+.1f} | ReLU = {relu(z_ej):.1f}")

### Qué es ReLU

**ReLU** (*Rectified Linear Unit*) es `max(0, z)`: devuelve cero para puntajes negativos y el puntaje tal cual para los positivos. Su gráfica es una recta horizontal pegada al cero que, al cruzar el origen, sube con pendiente 1.

Dos propiedades que la hacen popular: es trivial de calcular y **no se satura del lado positivo** (su derivada vale 1 ahí), así que el gradiente no se apaga mientras la neurona esté activa.

### Qué activación usar en la capa de salida: 3 casos

La salida depende del tipo de problema:

1. **Clasificación binaria** (¿maligno o benigno?): `sigmoid`, porque entrega una probabilidad entre 0 y 1.
2. **Regresión con valores positivos y negativos** (ej. cambio esperado del precio de una acción): `linear`, porque la respuesta no está acotada.
3. **Regresión con valores solo no negativos** (ej. precio de una casa): `ReLU`, que garantiza salida $\geq 0$ sin acotarla por arriba.

Elegir mal aquí no es un detalle estético: una sigmoide en una regresión de precios nunca podrá predecir más de 1, y una lineal en una clasificación puede devolver $-3$ o $7$ como si fueran probabilidades.

### Capas ocultas: ReLU es el estándar

Para las capas internas, la elección por defecto es **ReLU**, y la razón es práctica:

- Su gradiente vale 1 para $z > 0$: las neuronas activas aprenden a ritmo constante, sin el freno de la sigmoide saturada.
- Es muy barata de calcular (`max(0, z)`), lo que importa con millones de neuronas.
- Produce activaciones dispersas (muchas en cero), lo que suele simplificar el aprendizaje.

La sigmoide en capas ocultas fue la norma histórica, pero se abandonó en la práctica por el estancamiento del gradiente. `tanh` fue su reemplazo natural (centrada en cero) y hoy ReLU domina.

### Otras activaciones (mención breve)

Existen más y tienen nichos: **tanh** (como sigmoid pero entre $-1$ y $1$, centrada en cero), **Leaky ReLU** (deja pasar un poco los negativos para que no mueran), **softmax** (convierte un vector en probabilidades que suman 1, obligatoria en clasificación multiclase) y variantes modernas como **GELU** o **swish** en modelos grandes. Para estos labs, con sigmoid, lineal y ReLU se cubre lo esencial.

### Por qué se necesitan: sin no-linealidad la red colapsa

Si todas las capas usan activación lineal, apilarlas no agrega nada. Dos capas serían:

$$a^{[1]} = W^{[1]} x + b^{[1]}, \qquad a^{[2]} = W^{[2]} a^{[1]} + b^{[2]}$$

Sustituyendo: $a^{[2]} = (W^{[2]} W^{[1]}) x + (W^{[2]} b^{[1]} + b^{[2]})$. El producto de matrices es otra matriz y la combinación de sesgos es otro sesgo: **toda la red equivale a una sola capa lineal**, por más profundidad que tenga. Una red así no puede dibujar curvas; sería una regresión lineal carísima.

In [ ]:
import numpy as np

# Dos capas lineales encadenadas
def capa1(x):
    return 2.0 * x + 1.0

def capa2(a):
    return 3.0 * a - 2.0

def red_lineal(x):
    return capa2(capa1(x))

# La capa única equivalente: (3 * 2) x + (3 * 1 - 2)
def capa_unica(x):
    return 6.0 * x + 1.0

print(f"capa 1: a = 2x + 1 | capa 2: a = 3a - 2")
print(f"Red de 2 capas vs 1 capa equivalente:")
for x_ej in [0.0, 1.0, 2.0, -1.0]:
    print(f"x = {x_ej:+.1f}: red = {red_lineal(x_ej):+.1f} | capa única = {capa_unica(x_ej):+.1f} | ¿igual? {np.isclose(red_lineal(x_ej), capa_unica(x_ej))}")
print("Con activación lineal, la profundidad no aporta capacidad nueva.")

### Piezas lineales: cómo ReLU construye curvas con rectas

Una red con ReLU no dibuja una curva suave: **pega tramos rectos** (piecewise linear). Cada ReLU aporta un **codo**: antes de su punto de transición vale cero y no contribuye a la salida; al activarse, suma una recta nueva que cambia la pendiente del total.

Ejemplo con una recta base y dos neuronas:

- Base: 0.5x.
- Neurona A: 1.5 × max(0, x − 2). Se activa en x = 2.
- Neurona B: −2 × max(0, x − 4). Se activa en x = 4 y resta pendiente.

Sumando, la salida queda en tres tramos: antes de 2 pendiente 0.5; entre 2 y 4 pendiente 0.5 + 1.5 = 2; desde 4 pendiente 2 − 2 = 0 (plana). Los valores confirman el recorrido: f(0) = 0, f(2) = 1, f(3) = 3, f(4) = 5, f(5) = 5.

El detalle de "a veces después": si el peso de la ReLU es negativo, la neurona se activa antes del punto y se apaga después; el quiebre ocurre igual, solo cambia de lado. Muchos codos juntos producen una línea quebrada con tantos tramos como se quiera, y con suficientes tramos se aproxima cualquier curva.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Estilo para las gráficas
plt.style.use('ggplot')

x = np.linspace(-1, 6, 500)
base = 0.5 * x
a = 1.5 * np.maximum(0, x - 2)
b = -2.0 * np.maximum(0, x - 4)
total = base + a + b

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Panel izquierdo: los componentes por separado
ax1.plot(x, base, c='b', label='base: 0.5x')
ax1.plot(x, a, c='g', label='ReLU A: 1.5 max(0, x-2)')
ax1.plot(x, b, c='r', label='ReLU B: -2 max(0, x-4)')
ax1.set_title("Los componentes: dos rectas que esperan su turno")
ax1.set_xlabel("x")
ax1.set_ylabel("aporte")
ax1.legend()

# Panel derecho: la suma, con cada tramo de un color
m1 = x < 2
m2 = (x >= 2) & (x < 4)
m3 = x >= 4
ax2.plot(x[m1], total[m1], c='b', linewidth=3, label='tramo 1: pendiente 0.5')
ax2.plot(x[m2], total[m2], c='g', linewidth=3, label='tramo 2: pendiente 2')
ax2.plot(x[m3], total[m3], c='r', linewidth=3, label='tramo 3: pendiente 0')
ax2.axvline(2, c='k', linestyle=':', label='codo en x = 2')
ax2.axvline(4, c='gray', linestyle=':', label='codo en x = 4')
ax2.set_title("La suma: una línea quebrada de tres tramos")
ax2.set_xlabel("x")
ax2.legend()
plt.tight_layout()
plt.show()

# Pendiente medida en cada tramo y valores del ejemplo
for nombre, m in [("x < 2", m1), ("2 <= x < 4", m2), ("x >= 4", m3)]:
    xs, ys = x[m], total[m]
    pendiente = (ys[-1] - ys[0]) / (xs[-1] - xs[0])
    print(f"tramo {nombre}: pendiente = {pendiente:.2f}")
for x_ej in [0.0, 2.0, 3.0, 4.0, 5.0]:
    print(f"f({x_ej:.0f}) = {0.5*x_ej + 1.5*max(0, x_ej-2) - 2.0*max(0, x_ej-4):.1f}")

### Cierre

Regla práctica: **ReLU en las capas ocultas**, y en la salida según el problema — `sigmoid` para clasificación binaria, `linear` para regresión sin restricciones y `ReLU` para regresión no negativa. La no-linealidad es lo que hace que apilar capas tenga sentido: sin ella, la red entera es una sola recta. Lo que sigue: evaluar el modelo entrenado y cuidar el sobreajuste en TensorFlow.